In [ ]:
# Public packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import seaborn as sns
from tqdm import tqdm
import math
import os
import re

# Custom packages
from filter import FilterDF as fdf
from benchmarks import ParetoAnalysis as pa
from benchmarks import AccuracyCalculation as ac
from integrity_fixes import DataFixer as fix, DataExporter as exporter

In [ ]:
%load_ext autoreload
%autoreload 2

Load formatted data

In [ ]:
%store -r static_data_merged
%store -r sales_data_merged

Read data from parquet files

In [ ]:
# Check if the data is already imported
if 'static_data_merged' not in locals():

    # Retrieve filnames
    static_data_filenames = os.listdir(f"2_palate_data_parquet_cleaned")

    # Initialize a dictionary for the four parquet files of "static reference data"
    static_data = {}
    for filename in tqdm(static_data_filenames):

        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"2_palate_data_parquet_cleaned/{filename}")
            base_name = re.sub(r'\.parquet$', '', filename)
            static_data[base_name] = df

    # Rename and store
    static_data_merged = static_data.copy()
    %store static_data_merged


# Data already exists
else:
    static_data = static_data_merged.copy()

# Check if the data is already imported
if 'sales_data_merged' not in locals():

    # Retrieve filnames
    restaurant_sales_filenames = os.listdir(f"2_palate_data_parquet_cleaned/orders_item_level")

    # Initialize a dictionary for the 30 parquet files of "restaurant sales data"
    sales_and_menu_data = {}
    for filename in tqdm(restaurant_sales_filenames):
        
        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"2_palate_data_parquet_cleaned/orders_item_level/{filename}")
            location_id = re.sub(r'_sales_and_menu\.parquet$', '', filename)
            sales_and_menu_data[location_id] = df
    
    # Rename and store
    sales_data_merged = {}
    for loc_id, df in sales_and_menu_data.items():
        sales_data_merged[loc_id] = df.copy()
    %store sales_data_merged

# Data already exists
else:
    sales_and_menu_data = {}
    for loc_id, df in sales_data_merged.items():
        sales_and_menu_data[loc_id] = df.copy()


# Rename data for ease of use
before_after_details = static_data['before_after_details'].copy() # Promotional items (30 rows)
customers = static_data['customers'].copy() # Specific customer information: for matching customers with orders
items_tagged = static_data['items_tagged'].copy() # Menu items for all restaurants: for matching plant-based labels with orders
locations = static_data['locations'].copy() # Restaurant details (30 rows)
location_ids = list(sales_and_menu_data.keys())

In [ ]:
for loc_id in location_ids:
    df = df.assign(item_modifications = lambda df: df['item_modifications'].str.title())

Load formatted data

In [ ]:
%store -r static_data_merged_misclassified
%store -r sales_data_merged_misclassified

In [ ]:
# Check if the data is already imported
if 'static_data_merged_misclassified' not in locals():

    # Retrieve filnames
    static_data_filenames = os.listdir(f"2_palate_data_parquet_misclassified")

    # Initialize a dictionary for the four parquet files of "static reference data"
    static_data_misclassified = {}
    for filename in tqdm(static_data_filenames):

        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"2_palate_data_parquet_misclassified/{filename}")
            base_name = re.sub(r'\.parquet$', '', filename)
            static_data_misclassified[base_name] = df

    # Rename and store
    static_data_merged_misclassified = static_data_misclassified.copy()
    %store static_data_merged_misclassified


# Data already exists
else:
    static_data_misclassified = static_data_merged_misclassified.copy()

# Check if the data is already imported
if 'sales_data_merged_misclassified' not in locals():

    # Retrieve filnames
    restaurant_sales_filenames = os.listdir(f"2_palate_data_parquet_misclassified/orders_item_level")

    # Initialize a dictionary for the 30 parquet files of "restaurant sales data"
    sales_and_menu_data_misclassified = {}
    for filename in tqdm(restaurant_sales_filenames):
        
        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"2_palate_data_parquet_misclassified/orders_item_level/{filename}")
            location_id = re.sub(r'_sales_and_menu\.parquet$', '', filename)
            sales_and_menu_data_misclassified[location_id] = df
    
    # Rename and store
    sales_data_merged_misclassified = {}
    for loc_id, df in sales_and_menu_data_misclassified.items():
        sales_data_merged_misclassified[loc_id] = df.copy()
    %store sales_data_merged_misclassified

# Data already exists
else:
    sales_and_menu_data_misclassified = {}
    for loc_id, df in sales_data_merged_misclassified.items():
        sales_and_menu_data_misclassified[loc_id] = df.copy()


# Rename data for ease of use
before_after_details = static_data_misclassified['before_after_details'].copy() # Promotional items (30 rows)
customers = static_data_misclassified['customers'].copy() # Specific customer information: for matching customers with orders
items_tagged = static_data_misclassified['items_tagged'].copy() # Menu items for all restaurants: for matching plant-based labels with orders
locations = static_data_misclassified['locations'].copy() # Restaurant details (30 rows)
location_ids = list(sales_and_menu_data_misclassified.keys())

## Static Reference Data Exploration

### Retrieval

Retrieve the four static reference data frames

In [ ]:
# Promotional items (30 rows)
before_after_details = static_data_misclassified['before_after_details'].copy()

# Specific customer information: for matching customers with orders
customers = static_data_misclassified['customers'].copy()

# Menu items for all restaurants: for matching plant-based labels with orders
items_tagged = static_data_misclassified['items_tagged'].copy()

# Restaurant details (30 rows)
locations = static_data_misclassified['locations'].copy()

# List out locations
location_ids = list(sales_and_menu_data_misclassified.keys())

### Benchmarks

Pareto analysis

In [ ]:
list_of_percentiles = []
list_of_num_dishes = []
list_of_top_items = []

j = 0
startrow = 0 
startrow2 = 0

with pd.ExcelWriter('restaurant_samples.xlsx', engine='openpyxl') as writer:

    # Determine the number of menu items for all 30 restaurants
    for loc_id in location_ids:

        header = True if startrow == 0 else False

        # Filter to the current restaurant and without alcohol
        menu_items = fdf(items_tagged).filter('location_id', loc_id).drop_duplicates('id')
        sales_and_menu_no_alcohol = fdf(merged_sales_and_menu[loc_id]).filter('dish_category', 'Alcohol', exclude=True)

        # 'entries', 'quantity', 'sales'
        target_percentage = .8
        percentiles = pa.create_percentiles(sales_and_menu_no_alcohol, metric='quantity')
        index = pa.num_items_to_cover_certain_percentage(sales_and_menu_no_alcohol, metric='quantity', percentile=target_percentage)
        
        list_of_percentiles.append(percentiles)
        list_of_num_dishes.append((index+1, percentiles.size))
        
        top_items = percentiles.iloc[:index+1]
        top_items_df = pd.DataFrame(top_items)
        top_items_df.reset_index(inplace=True)
        top_items_df['location_id'] = loc_id
        menu_lower = menu_items.copy()
        menu_lower['item_name'] = menu_lower['item_name'].str.lower()
        menu_lower.drop_duplicates(['item_name', 'location_id'], inplace=True) 
        top_items_with_id = pd.merge(top_items_df, menu_lower, on=['location_id', 'item_name'], how='inner').loc[:,['id', 'item_name', 'location_id']]
        list_of_top_items.append(top_items_with_id)

        items_tagged_copy = items_tagged.copy()
        menu_no_alcohol_all = fdf(items_tagged_copy).filter('dish_category', 'Alcohol', exclude=True)
        menu_no_alcohol = fdf(menu_no_alcohol_all).filter('location_id', loc_id)
        menu_no_alcohol['item_name'] = menu_no_alcohol['item_name'].str.lower()
        menu_no_alcohol.drop_duplicates('item_name', inplace=True)
        menu_no_alcohol['item_name'] = menu_no_alcohol['item_name'].str.capitalize()

        print(loc_id)
        is_ten = False
        n_sample = 10
        while not is_ten:
            sample = ac.calculate_accuracy(top_items, menu_no_alcohol, n_sample=n_sample).loc[:,['item_name', 'item_type', 'dish_category', 'is_plant_based', 'ingredients']]
            if sample.shape[0] == 10 or n_sample > 20:
                is_ten = True
            n_sample += 1

        pd.DataFrame(np.array([[f'Restaurant {j+1}'], [loc_id]]).T, index=['']).to_excel(writer, sheet_name='Sheet1', startrow=startrow, header=False, index=True)
        startrow += 1
    
        sample.to_excel(writer, sheet_name='Sheet1', startrow=startrow, header=header, index=True)
        top_items_with_id.to_excel(writer, sheet_name='Sheet2', startrow=startrow2, header=header, index=False)
        if header:
            startrow += 1
        startrow += sample.shape[0] + 2
        startrow2 += top_items_with_id.shape[0]
        j += 1


In [ ]:
checking = pd.merge(items_tagged_new, pd.concat(list_of_top_items), left_on = 'item_name', right_on = 'item_name', how='outer', indicator=True)
checking[checking['_merge'] == 'left_only'].shape[0], checking[checking['_merge'] == 'right_only'].shape[0]
checking[checking['_merge'] == 'left_only']

In [ ]:
sample_mislabel_list = [
{'restaurant_id':location_ids[0],
'true_positives': 3, 
'false_positives': 0,
'true_negatives': 5,
'false_negatives': 1,
'ambiguous': 1},

{'restaurant_id':location_ids[1],
'true_positives': 1, 
'false_positives': 1,
'true_negatives': 4,
'false_negatives': 2,
'ambiguous': 2},

{'restaurant_id':location_ids[2],
'true_positives': 1, 
'false_positives': 2,
'true_negatives': 2,
'false_negatives': 0,
'ambiguous': 5},

{'restaurant_id':location_ids[3],
'true_positives': 1, 
'false_positives': 2,
'true_negatives': 4,
'false_negatives': 1,
'ambiguous': 2},

{'restaurant_id':location_ids[4],
'true_positives': 4, 
'false_positives': 0,
'true_negatives': 5,
'false_negatives': 0,
'ambiguous': 1},

{'restaurant_id':location_ids[5],
'true_positives': 2, 
'false_positives': 1,
'true_negatives': 4,
'false_negatives': 0,
'ambiguous': 3},

{'restaurant_id':location_ids[6],
'true_positives': 1, 
'false_positives': 0,
'true_negatives': 2,
'false_negatives': 0,
'ambiguous': 7},   # I have no idea what this stuff is

{'restaurant_id':location_ids[7],
'true_positives': 1, 
'false_positives': 0,
'true_negatives': 2,
'false_negatives': 0,
'ambiguous': 7},

{'restaurant_id':location_ids[8],
'true_positives': 4, 
'false_positives': 0,
'true_negatives': 3,
'false_negatives': 0,
'ambiguous': 3},

{'restaurant_id':location_ids[9],
'true_positives': 4, 
'false_positives': 0,
'true_negatives': 1,
'false_negatives': 5,
'ambiguous': 0},

{'restaurant_id':location_ids[10],
'true_positives': 7, 
'false_positives': 0,
'true_negatives': 2,
'false_negatives': 0,
'ambiguous': 1},

{'restaurant_id':location_ids[11],
'true_positives': 4, 
'false_positives': 0,
'true_negatives': 2,
'false_negatives': 0,
'ambiguous': 4},

{'restaurant_id':location_ids[12],
'true_positives': 7,    # probably more like ambigious given coffee usually has creamer
'false_positives': 3,
'true_negatives': 0,
'false_negatives': 0,
'ambiguous': 0},

{'restaurant_id':location_ids[13],
'true_positives': 2, 
'false_positives': 1,
'true_negatives': 7,
'false_negatives': 0,
'ambiguous': 0},

{'restaurant_id':location_ids[14],
'true_positives': 6, 
'false_positives': 0,
'true_negatives': 2,
'false_negatives': 0,
'ambiguous': 2},

{'restaurant_id':location_ids[15],
'true_positives': 2, 
'false_positives': 0,
'true_negatives': 5,
'false_negatives': 1,
'ambiguous': 2},

{'restaurant_id':location_ids[16],
'true_positives': 3, 
'false_positives': 0,
'true_negatives': 6,
'false_negatives': 1,
'ambiguous': 0},

{'restaurant_id':location_ids[17],
'true_positives': 0, 
'false_positives': 1,
'true_negatives': 7,
'false_negatives': 0,
'ambiguous': 2},

{'restaurant_id':location_ids[18],
'true_positives': 2, 
'false_positives': 1,
'true_negatives': 4,
'false_negatives': 2,
'ambiguous': 1}, # 'Single.'

{'restaurant_id':location_ids[19],
'true_positives': 2, 
'false_positives': 3,
'true_negatives': 4,
'false_negatives': 0,
'ambiguous': 1},   # credit card fee?

{'restaurant_id':location_ids[20],
'true_positives': 3, 
'false_positives': 0,
'true_negatives': 2,
'false_negatives': 0,
'ambiguous': 5},  # "custom amount"?

{'restaurant_id':location_ids[21],
'true_positives': 1, 
'false_positives': 1,
'true_negatives': 6,
'false_negatives': 0,
'ambiguous': 2},

{'restaurant_id':location_ids[22],    # only 8 items for top 90%
'true_positives': 2, # we'll count impossible melt as yes
'false_positives': 1,
'true_negatives': 3,
'false_negatives': 0,
'ambiguous': 1},  # maybe 2 ambigious if we count gold standard kale with egg ingredient

{'restaurant_id':location_ids[23],
'true_positives': 4,  # sausage, egg, cheese (v) ? 
'false_positives': 4,
'true_negatives': 2,
'false_negatives': 0,
'ambiguous': 0},

{'restaurant_id':location_ids[24],
'true_positives': 5, 
'false_positives': 1,
'true_negatives': 2,
'false_negatives': 0,
'ambiguous': 2},

{'restaurant_id':location_ids[25],
'true_positives': 4, 
'false_positives': 1,
'true_negatives': 5,
'false_negatives': 0,
'ambiguous': 0},

{'restaurant_id':location_ids[26],  # only has like 10 items in top 90%
'true_positives': 2, 
'false_positives': 0,
'true_negatives': 4,
'false_negatives': 3,
'ambiguous': 1},  # "single"

{'restaurant_id':location_ids[27],   # more alcohol not labeled in alcohol category
'true_positives': 6, 
'false_positives': 0,
'true_negatives': 1,
'false_negatives': 0, 
'ambiguous': 3},   # "egift card", "foreland stolen artifacts"

{'restaurant_id':location_ids[28],
'true_positives': 3, 
'false_positives': 3,
'true_negatives': 2,
'false_negatives': 0,
'ambiguous': 2},  # "card"

{'restaurant_id':location_ids[29],
'true_positives': 4, 
'false_positives': 0,
'true_negatives': 5,
'false_negatives': 0,
'ambiguous': 1},
]

In [ ]:
df.head()

In [ ]:
import pypalettes
import highlight_text

df = pd.read_csv("https://raw.githubusercontent.com/holtzy/The-Python-Graph-Gallery/master/static/data/simple-treemap.csv")

# create a color palette
cmap = pypalettes.load_cmap('Acadia')
category_codes, unique_categories = pd.factorize(df['parent'])
colors = [cmap(code) for code in category_codes]

# create a treemap
fig, ax = plt.subplots(figsize=(10,10))
ax.set_axis_off()
squarify.plot(
   sizes=df["value"],
   label=df["name"],
   color=colors,
   text_kwargs={'color':'white'},
   pad=True,
   ax=ax
)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import squarify  # For treemap layout
from PIL import Image
import numpy as np

# Load textures for each category
textures = {
    "True Positive": Image.open('leaf_texture.jpg'),
    "False Positive": Image.open('meat_pattern.jpg'),
    "True Negative": Image.open('meat_pattern.jpg'),
    "False Negative": Image.open('leaf_texture.jpg')
}

# Flattened data list for squarify
labels = ["True Negative", "True Positive", "False Positive", "False Negative"]
values = [40, 20, 50, 10]  # Corresponding sizes for each category
confusion_df = pd.DataFrame(data=zip(values,labels), columns=['value','label'])

# Use squarify to calculate the treemap layout
#rects = squarify.normalize_sizes(values, 100, 100)
#squarified_rects = squarify.squarify(rects, 0, 0, 100, 100)

fig, ax = plt.subplots(figsize=(8, 8))
squarify.plot(sizes=confusion_df['value'], 
              label=confusion_df['label'], 
              pad=True,
              ax=ax)


# for i, rect in enumerate(squarified_rects):
#     label = labels[i]
#     x, y, width, height = rect['x'], rect['y'], rect['dx'], rect['dy']
    
#     # Draw a rectangle with a border
#     ax.add_patch(plt.Rectangle((x, y), width, height, edgecolor='black', facecolor='none'))
    
#     # Overlay texture
#     texture_img = textures[label]
#     ax.imshow(texture_img, extent=(x, x + width, y, y + height), alpha=0.3)

#     # Add a label at the center of each rectangle
#     ax.text(x + width / 2, y + height / 2, label, ha="center", va="center", color="black")

# Turn off the axes
ax.axis('off')
plt.show()

In [ ]:
Image.open('leaf_texture.jpg')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

# Load textures for each category
textures = {
    "True Positive": Image.open('leaf_texture.jpg'),
    "False Positive": Image.open('meat_pattern.jpg'),
    "True Negative": Image.open('meat_pattern.jpg'),
    "False Negative": Image.open('leaf_texture.jpg')
}

# Define the positions and sizes for each category
# Assuming the plot space is divided based on the specified values
# Adjusted proportions to sum up to 1
data = {
    "True Positive": {"pos": (0, 0.5), "size": (0.5, 0.5)},   # Top-left
    "False Negative": {"pos": (0.5, 0.5), "size": (0.5, 0.5)}, # Top-right
    "False Positive": {"pos": (0, 0), "size": (0.5, 0.5)},    # Bottom-left
    "True Negative": {"pos": (0.5, 0), "size": (0.5, 0.5)}    # Bottom-right
}

# Create the plot
fig, ax = plt.subplots(figsize=(8, 8))

# Draw each rectangle according to the manual layout
for label, info in data.items():
    x, y = info["pos"]
    width, height = info["size"]
    
    # Draw the rectangle with a border
    ax.add_patch(patches.Rectangle((x, y), width, height, edgecolor='black', facecolor='none'))
    
    # Apply the texture as an overlay within the rectangle
    texture_img = textures[label]
    ax.imshow(texture_img, extent=(x, x + width, y, y + height), alpha=0.3)
    
    # Add the label in the center of each rectangle
    ax.text(x + width / 2, y + height / 2, label, ha="center", va="center", color="black", fontsize=12, weight="bold")

# Turn off the axes for a clean look
ax.axis('off')
plt.show()


Accuracy Calculation

In [ ]:
sample_mislabel_counts = pd.DataFrame(sample_mislabel_list)

In [ ]:
sample_mislabel_counts[['false_positives', 'false_negatives']].sum().sum()

In [ ]:
accuracy = sample_mislabel_counts[['true_positives', 'true_negatives']].sum().sum() / sample_mislabel_counts.sum()[1:].sum()
error_rate = sample_mislabel_counts[['false_positives', 'false_negatives']].sum().sum() / sample_mislabel_counts.sum()[1:].sum()
ambiguous = sample_mislabel_counts['ambiguous'].sum() / sample_mislabel_counts.sum()[1:].sum()
sensitivity = sample_mislabel_counts['true_positives'].sum() / sample_mislabel_counts[['true_positives', 'false_negatives']].sum().sum()
specificity = sample_mislabel_counts['true_negatives'].sum() / sample_mislabel_counts[['false_positives', 'true_negatives']].sum().sum()
precision = sample_mislabel_counts['true_positives'].sum() / sample_mislabel_counts[['true_positives', 'false_positives']].sum().sum()
negative_predictive_value = sample_mislabel_counts['true_negatives'].sum() / sample_mislabel_counts[['true_negatives', 'false_negatives']].sum().sum()

"General split:", accuracy, error_rate, ambiguous, "More details: ", sensitivity, specificity, precision, negative_predictive_value

Fixing capitalization errors in merging

In [ ]:
# Too difficult, just lower everything for counting errors

# merge_mismatches= [] 
# for loc_id in tqdm(location_ids):
#     restaurant_df = fdf(items_tagged).filter('location_id', loc_id)
#     items = restaurant_df['item_name'].tolist()
#     restaurant_merge_mismatches_list = []
#     for item in items:
#         capitalization_variation_counts = fdf(restaurant_df).filter('item_name', item)
#         restaurant_merge_mismatches_list.append((loc_id, item, capitalization_variation_counts.shape[0] - 1))
#         restaurant_merge_mismatches = pd.DataFrame(restaurant_merge_mismatches_list)
#     merge_mismatches.append((loc_id, restaurant_merge_mismatches[2].any()))

In [ ]:
list_of_new_samples = []
for loc_id in location_ids:
    df = fdf(items_tagged_new).filter('location_id', loc_id).copy()
    list_of_new_samples.append(ac.calculate_accuracy(df['item_name'].value_counts(), df))

In [ ]:
new_samples = pd.concat(list_of_new_samples)

In [ ]:
my_labels_for_new_samples_list = []
for i, row in new_samples.iterrows():
    my_label = input(row['item_name'] + ': Is it vegan?')
    my_labels_for_new_samples_list.append((row['location_id'], row['item_name'], my_label))


In [ ]:
my_labels_for_new_samples = pd.DataFrame(my_labels_for_new_samples_list, columns=['location_id','item_name','my_label'])
my_labels_for_new_samples['my_label'] = my_labels_for_new_samples['my_label'].str.capitalize()

In [ ]:
new_sample_comparison = pd.merge(new_samples, my_labels_for_new_samples, on=['location_id','item_name'], how='left').drop(columns=['id','brand'])

In [ ]:
probably_wrong = ['veg supreme', 'side slaw', 'hot chocolate']

maybe_wrong = ['chips', 'gold standard sandwich', 'americano', 'tea latte', 'milk substitute', 'kettle brand potato chips']

In [ ]:
%store new_sample_comparison